# Notebook 14 – Performance Optimization

## Introduction
Performance optimization means improving the speed and memory efficiency of code.
Pandas provides several techniques to process large datasets efficiently.
In this notebook, we will explore vectorization, apply() vs loops,
memory optimization, categorical data types, and efficient Pandas practices.

In [1]:
import pandas as pd
import numpy as np
import time

## 1. Vectorization

Vectorization performs operations on entire arrays or Series without explicit Python loops.
It is usually faster than processing values one by one.
Pandas and NumPy support vectorized arithmetic and comparisons.
Vectorization is useful when processing large datasets.

In [2]:
numbers = pd.Series(range(1, 100001))

start = time.perf_counter()
vectorized_result = numbers * 2
vectorized_time = time.perf_counter() - start

print("First 5 results:")
print(vectorized_result.head())
print("Vectorized time:", vectorized_time)

First 5 results:
0     2
1     4
2     6
3     8
4    10
dtype: int64
Vectorized time: 0.0014285000506788492


In [3]:
start = time.perf_counter()
loop_result = [number * 2 for number in numbers]
loop_time = time.perf_counter() - start

print("First 5 results:", loop_result[:5])
print("Loop time:", loop_time)

First 5 results: [2, 4, 6, 8, 10]
Loop time: 0.025463900063186884


## 2. apply() vs Loops

The apply() method applies a function to elements of a Series or rows or columns of a DataFrame.
Loops process elements one by one using constructs such as for.
Vectorized operations are often faster than both approaches for simple calculations.
Use apply() when a custom function is needed and vectorization is not practical.

In [4]:
df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie", "David"],
    "Salary": [30000, 40000, 50000, 60000]
})

# Using apply()
df["Bonus_Apply"] = df["Salary"].apply(lambda salary: salary * 0.10)

print(df)

      Name  Salary  Bonus_Apply
0    Alice   30000       3000.0
1      Bob   40000       4000.0
2  Charlie   50000       5000.0
3    David   60000       6000.0


In [5]:
bonuses = []

for salary in df["Salary"]:
    bonuses.append(salary * 0.10)

df["Bonus_Loop"] = bonuses

print(df)

      Name  Salary  Bonus_Apply  Bonus_Loop
0    Alice   30000       3000.0      3000.0
1      Bob   40000       4000.0      4000.0
2  Charlie   50000       5000.0      5000.0
3    David   60000       6000.0      6000.0


In [6]:
df["Bonus_Vectorized"] = df["Salary"] * 0.10

print(df)

      Name  Salary  Bonus_Apply  Bonus_Loop  Bonus_Vectorized
0    Alice   30000       3000.0      3000.0            3000.0
1      Bob   40000       4000.0      4000.0            4000.0
2  Charlie   50000       5000.0      5000.0            5000.0
3    David   60000       6000.0      6000.0            6000.0


## 3. Memory Optimization

Memory optimization reduces the amount of RAM used by a DataFrame.
We can inspect memory usage with memory_usage(deep=True).
Appropriate numeric data types can reduce memory consumption.
This is particularly useful when working with large datasets.

In [7]:
df_memory = pd.DataFrame({
    "Age": np.array([22, 25, 30, 35, 40], dtype=np.int64),
    "Salary": np.array([30000, 40000, 50000, 60000, 70000], dtype=np.int64)
})

print("Before optimization:")
print(df_memory.dtypes)
print(df_memory.memory_usage(deep=True))
print("Total bytes:", df_memory.memory_usage(deep=True).sum())

Before optimization:
Age       int64
Salary    int64
dtype: object
Index     132
Age        40
Salary     40
dtype: int64
Total bytes: 212


In [8]:
df_memory["Age"] = df_memory["Age"].astype("int8")
df_memory["Salary"] = pd.to_numeric(
    df_memory["Salary"], downcast="integer"
)

print("\nAfter optimization:")
print(df_memory.dtypes)
print(df_memory.memory_usage(deep=True))
print("Total bytes:", df_memory.memory_usage(deep=True).sum())


After optimization:
Age        int8
Salary    int32
dtype: object
Index     132
Age         5
Salary     20
dtype: int64
Total bytes: 157


## 4. Category Data Type

The category data type stores repeated text values using a set of categories.
It can reduce memory usage when a column has relatively few unique values.
It is useful for columns such as city, department, product type, or status.
Categorical columns can also be useful for grouping and analysis.

In [9]:
df_category = pd.DataFrame({
    "City": ["Hyderabad", "Chennai", "Hyderabad",
             "Mumbai", "Chennai", "Hyderabad"] * 1000
})

print("Before conversion:")
print(df_category["City"].dtype)
print(
    "Memory:",
    df_category["City"].memory_usage(deep=True),
    "bytes"
)

Before conversion:
str
Memory: 95132 bytes


In [10]:
df_category["City"] = df_category["City"].astype("category")

print("\nAfter conversion:")
print(df_category["City"].dtype)
print(
    "Memory:",
    df_category["City"].memory_usage(deep=True),
    "bytes"
)

print(df_category["City"].value_counts())


After conversion:
category
Memory: 6179 bytes
City
Hyderabad    3000
Chennai      2000
Mumbai       1000
Name: count, dtype: int64


## 5. Efficient Pandas Practices

Use vectorized operations instead of explicit loops when possible.
Select only the columns needed for analysis.
Choose appropriate data types to reduce memory consumption.
Avoid unnecessary copies and repeated calculations.
For large CSV files, consider reading only required columns or using chunksize.

In [11]:
employees = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie", "David"],
    "Department": ["IT", "HR", "IT", "Finance"],
    "Salary": [50000, 35000, 60000, 45000],
    "Age": [25, 30, 28, 35]
})

# Select only required columns
selected = employees[["Name", "Salary"]]

# Filter using a vectorized condition
high_salary = employees.loc[employees["Salary"] > 40000]

# Calculate a new column using vectorization
employees["Annual_Bonus"] = employees["Salary"] * 0.10

print("Selected columns:")
print(selected)

print("\nEmployees earning above 40000:")
print(high_salary)

print("\nUpdated employee data:")
print(employees)

Selected columns:
      Name  Salary
0    Alice   50000
1      Bob   35000
2  Charlie   60000
3    David   45000

Employees earning above 40000:
      Name Department  Salary  Age
0    Alice         IT   50000   25
2  Charlie         IT   60000   28
3    David    Finance   45000   35

Updated employee data:
      Name Department  Salary  Age  Annual_Bonus
0    Alice         IT   50000   25        5000.0
1      Bob         HR   35000   30        3500.0
2  Charlie         IT   60000   28        6000.0
3    David    Finance   45000   35        4500.0


## Practical Example – Salary Calculation

We will calculate bonuses using vectorization and compare the result with a loop.
The purpose is to understand alternative approaches and verify that the results match.

In [13]:
salaries = pd.Series([30000, 40000, 50000, 60000, 70000])

# Vectorized calculation
bonus_vectorized = salaries * 0.10

# Loop calculation
bonus_loop = []

for salary in salaries:
    bonus_loop.append(salary * 0.10)

print("Vectorized bonuses:", bonus_vectorized.tolist())
print("Loop bonuses:", bonus_loop)

print(
    "Results match:",
    np.allclose(bonus_vectorized.to_numpy(), bonus_loop)
)

Vectorized bonuses: [3000.0, 4000.0, 5000.0, 6000.0, 7000.0]
Loop bonuses: [3000.0, 4000.0, 5000.0, 6000.0, 7000.0]
Results match: True
